# Sample Builder Outputs EDA

This notebook explores how SampleBuilder configurations (provided through datamodules) behave on the
TACO trace datasets (a selected set of shards, or all shards). We measure what kind of samples are
planned for generation, and what kind of samples are actually generated at runtime (given trace data
constraints).

The notebook supports two modes for instantiating the datamodule:

1. **Option A, from experiment config**: load the datamodule exactly as it would be configured in an
   actual hf_trainer experiment by specifying an experiment config name (e.g., `"original/v0_rl"`).
   This ensures the analysis uses the same data configuration as the real experiment.
2. **Option B, manual configuration**: manually specify dataset paths, datamodule type, and other
   parameters for more flexible exploration.


In [ ]:
import collections
import random
import re
import types
import typing

import IPython.display as ipy_display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch.utils.data

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.organisms.datamodules.keywords as keywords
import pyine.organisms.datamodules.keywords_configs as keywords_configs
import pyine.organisms.datamodules.samples as sample_utils
import pyine.organisms.datamodules.samples.keyword_ops as kw_ops
import pyine.organisms.datamodules.shortcuts as shortcuts
import pyine.organisms.datamodules.shortcuts_configs as shortcuts_configs
import pyine.utils.notebooks
import pyine.utils.pydantic
import pyine.utils.reprod

pyine.utils.notebooks.setup_notebook_plotting()
pyine.utils.reprod.entrypoint_setup()

In [ ]:
# ------------ CHANGE THESE SETTINGS IF NEEDED ------------

# option A: load datamodule from experiment config (set to None to use option B instead)
# use the experiment name as it would be passed to hf_trainer (e.g., "original/v0_rl")
experiment_config_name: str | None = None  # e.g., "original/v0_rl"
experiment_config_overrides: list[str] | None = None  # e.g., ["runtime.seed=42"]

# option B: manual datamodule configuration (used when experiment_config_name is None)
source_dataset_name = "TACO"  # by default, we target the TACO 10s10t traces dataset
trace_dataset_pattern = "v1.5/10s10t.*of000026.*.lmdb"  # specific pattern for the v1 dataset
expected_part_count = 26  # given the 500-problem-chunk split used for the v1 dataset
selected_part_indices: int | list[int] = list(range(26))  # list of zero-based indices
target_datamodule = "shortcuts"
use_hybrid_sample_transforms = True
counterfactual_eval_strategy = True
seed = 0

# common settings (used for both modes)
max_samples_per_subset = None  # None = no maximum
epoch = 0
# ---------------------------------------------------------

In [ ]:
# prepare the target datamodule based on the selected configuration mode
kw_detector = types.SimpleNamespace(has_keyword=lambda x: None)  # default: no keyword detection

if experiment_config_name is not None:  # option A: load datamodule from experiment config
    print(f"loading datamodule from experiment config: {experiment_config_name}")
    if experiment_config_overrides:
        print(f"  with overrides: {experiment_config_overrides}")
    datamodule = pyine.utils.notebooks.load_datamodule_from_hf_trainer_config(
        experiment_name=experiment_config_name,
        overrides=experiment_config_overrides,
    )
    # determine target subsets and datamodule type from the loaded config
    dm_config = datamodule.config
    target_subsets = list(dm_config.subset_names)
    # check if this is a keywords datamodule to set up keyword detection
    if isinstance(datamodule, keywords.KeywordBiasDataModule):
        target_datamodule = "keywords"
        print(f"datamodule targets the following keyword: {datamodule.keyword}")
        kw_detector = kw_ops.KeywordDetector(keyword=datamodule.keyword)
    else:
        target_datamodule = "shortcuts"
else:  # option B: manual datamodule configuration
    dataset_split_file_path = pyine.data.utils.splits.get_dataset_split_file_path(source_dataset_name)
    dataset_paths = sorted(
        pyine.data.traces.dataset_utils.get_matching_dataset_paths(
            source_dataset_name=source_dataset_name,
            pattern=trace_dataset_pattern,
        )
    )
    if isinstance(selected_part_indices, int):
        selected_part_indices = [selected_part_indices]
    assert isinstance(selected_part_indices, list) and len(selected_part_indices) > 0, "missing shard selection"
    assert all(0 <= idx < len(dataset_paths) for idx in selected_part_indices), "invalid shard selection"
    dataset_paths = [dataset_paths[idx] for idx in selected_part_indices]
    print("will perform analysis on the following dataset shards:")
    for path in dataset_paths:
        print(f"- {path}")
    supported_datamodules = ["keywords", "shortcuts"]
    if target_datamodule not in supported_datamodules:
        raise ValueError(f"unsupported datamodule: {target_datamodule}; pick one of {supported_datamodules}")
    elif target_datamodule == "shortcuts":
        default_dm_config = shortcuts_configs.get_datamodule_config(
            lmdb_paths=dataset_paths,
            split_file_path=dataset_split_file_path,
            seed=seed,
            use_hybrid_sample_transforms=use_hybrid_sample_transforms,
            as_pydantic=True,
            min_samples_hinted=0,  # to keep the notebook working even if we lack augmented data
            evaluation_strategy="counterfactual" if counterfactual_eval_strategy else "hint_presence_split",
        )
        datamodule = shortcuts.ShortcutBiasDataModule(default_dm_config)
        datamodule._clear_prepared_metadata()  # to avoid stale cache issues...
        print("performing datamodule preparation + setup...")
        datamodule.prepare_data()
        datamodule.setup()
        target_subsets = list(datamodule.config.subset_names)
    elif target_datamodule == "keywords":
        default_dm_config = keywords_configs.get_datamodule_config(
            lmdb_paths=dataset_paths,
            split_file_path=dataset_split_file_path,
            seed=seed,
            use_hybrid_sample_transforms=use_hybrid_sample_transforms,
            as_pydantic=True,
            evaluation_strategy="counterfactual" if counterfactual_eval_strategy else "keyword_presence_split",
        )
        datamodule = keywords.KeywordBiasDataModule(default_dm_config)
        datamodule._clear_prepared_metadata()  # to avoid stale cache issues...
        print("performing datamodule preparation + setup...")
        datamodule.prepare_data()
        datamodule.setup()
        target_subsets = list(datamodule.config.subset_names)
        print(f"datamodule targets the following keyword: {datamodule.keyword}")
        kw_detector = kw_ops.KeywordDetector(keyword=datamodule.keyword)
    else:
        raise NotImplementedError(f"missing impl for datamodule: {target_datamodule}")

for subset_name in target_subsets:
    parser = datamodule.get_parser(subset_name)
    if hasattr(parser, "set_epoch"):
        parser.set_epoch(epoch)
    print(f"- {subset_name} parser will generate {len(parser)} samples per epoch")
print("datamodule ready-to-go!")

In [ ]:
pyine.utils.notebooks.display_pydantic_config(datamodule.config, title="Datamodule Configuration")

In [ ]:
builder_map: dict[str, torch.utils.data.Dataset] = {}
sample_rows: list[dict[str, typing.Any]] = []
for subset_name in target_subsets:
    parser = datamodule.get_parser(subset_name)
    if not len(parser):
        print(f"Skipping {subset_name}: no samples found.")
        continue
    builder_map[subset_name] = parser
    rows = pyine.utils.notebooks.collect_sample_metadata_rows(
        builder=parser,
        subset_name=subset_name,
        max_samples=max_samples_per_subset,
        keyword_detector=kw_detector.has_keyword,
    )
    sample_rows.extend(rows)

if sample_rows:
    print(f"sample collection complete (total samples: {len(sample_rows)})")
    samples_df = pd.DataFrame(sample_rows)
else:
    samples_df = pd.DataFrame()
    print("no samples collected; ensure the dataset is available and contains traces")

ipy_display.display(samples_df)

In [ ]:
found_subsets = samples_df["subset"].unique()
for subset in found_subsets:
    subset_df = samples_df[samples_df["subset"] == subset]
    print(f"subset '{subset}' has {len(subset_df):,} samples:")
    print(f"\t{len(subset_df['identifier']):,} unique identifiers")
    print(f"\t{subset_df['code_line_count'].mean():.1f} average line count")
    print(f"\t{subset_df['trace_step_count'].mean():.1f} average trace step count")
    print(f"\t{subset_df['inputs_char_length'].mean():.1f} average input length")
    percent_no_input = (subset_df["inputs_char_length"] == 0).mean() * 100
    print(f"\t{percent_no_input:.2f}% of samples have no inputs")
    print(f"\t{subset_df['expected_output_char_length'].mean():.1F} average expected output length")
    percent_falsy = subset_df["expected_output_falsy"].mean() * 100
    print(f"\t{percent_falsy:.2f}% of samples have empty expected outputs")

In [ ]:
# analyze sample ID overlap between subsets
if samples_df.empty:
    print("skipped; no data to analyze")
else:
    subset_names = list(samples_df["subset"].unique())
    subset_ids: dict[str, set[str]] = {}
    subset_base_ids: dict[str, set[str]] = {}
    subset_family_ids: dict[str, set[str]] = {}
    for subset in subset_names:
        subset_df = samples_df[samples_df["subset"] == subset]
        subset_ids[subset] = set(subset_df["identifier"])
        subset_base_ids[subset] = set(subset_df["base_id"])
        subset_family_ids[subset] = set(subset_df["family_id"])

    # build overlap matrices (exact IDs, base IDs, and family IDs)
    n_subsets = len(subset_names)
    exact_overlap = np.zeros((n_subsets, n_subsets), dtype=int)
    base_overlap = np.zeros((n_subsets, n_subsets), dtype=int)
    family_overlap = np.zeros((n_subsets, n_subsets), dtype=int)
    for row_idx, subset_a in enumerate(subset_names):
        for col_idx, subset_b in enumerate(subset_names):
            exact_overlap[row_idx, col_idx] = len(subset_ids[subset_a] & subset_ids[subset_b])
            base_overlap[row_idx, col_idx] = len(subset_base_ids[subset_a] & subset_base_ids[subset_b])
            family_overlap[row_idx, col_idx] = len(subset_family_ids[subset_a] & subset_family_ids[subset_b])

    # create DataFrames for display
    exact_df = pd.DataFrame(exact_overlap, index=subset_names, columns=subset_names)
    base_df = pd.DataFrame(base_overlap, index=subset_names, columns=subset_names)
    family_df = pd.DataFrame(family_overlap, index=subset_names, columns=subset_names)

    # display titles above the figure
    ipy_display.display(
        ipy_display.Markdown(
            "**Left: Exact ID Overlap** -- **Center: Base ID Overlap (ignoring ::suffix)** "
            "-- **Right: Family ID Overlap (augmentless trace ID)**"
        )
    )

    # plot all three matrices side by side
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    matrices = [exact_overlap, base_overlap, family_overlap]
    titles = ["Exact ID", "Base ID", "Family ID"]

    for ax, matrix, title in zip(axes, matrices, titles, strict=False):
        im = ax.imshow(matrix, cmap="YlGnBu")
        ax.set_xticks(range(n_subsets))
        ax.set_yticks(range(n_subsets))
        ax.set_xticklabels(subset_names, rotation=45, ha="right")
        ax.set_yticklabels(subset_names)
        ax.set_title(title)
        for row_idx in range(n_subsets):
            for col_idx in range(n_subsets):
                value = matrix[row_idx, col_idx]
                text_color = "white" if value > matrix.max() * 0.6 else "black"
                ax.text(col_idx, row_idx, f"{value:,}", ha="center", va="center", color=text_color, fontsize=9)
        fig.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.show()

    # also print the DataFrames for easy reference
    print("Exact ID overlap counts:")
    ipy_display.display(exact_df)
    print("\nBase ID overlap counts (ignoring ::suffix):")
    ipy_display.display(base_df)
    print("\nFamily ID overlap counts (augmentless trace ID):")
    ipy_display.display(family_df)

In [ ]:
# sanity check: problem ID overlap between subsets (should be zero between train/valid/test)
if samples_df.empty:
    print("skipped; no data to analyze")
else:

    def extract_problem_id(identifier: str) -> str | None:
        """Extract problem ID (e.g., 'p000001') from a sample identifier."""
        match = re.search(r"(p\d+)", identifier)
        return match.group(1) if match else None

    # only check primary subsets (train, valid, test) - derived subsets inherit from parents
    primary_subsets = [s for s in samples_df["subset"].unique() if s in ("train", "valid", "test")]
    subset_problem_ids: dict[str, set[str]] = {}
    for subset in primary_subsets:
        ids = samples_df[samples_df["subset"] == subset]["identifier"]
        problem_ids = {extract_problem_id(id_) for id_ in ids}
        problem_ids.discard(None)
        subset_problem_ids[subset] = problem_ids

    # build and display overlap matrix
    n = len(primary_subsets)
    overlap_matrix = np.zeros((n, n), dtype=int)
    for i, subset_a in enumerate(primary_subsets):
        for j, subset_b in enumerate(primary_subsets):
            overlap_matrix[i, j] = len(subset_problem_ids[subset_a] & subset_problem_ids[subset_b])

    overlap_df = pd.DataFrame(overlap_matrix, index=primary_subsets, columns=primary_subsets)
    print("Problem ID overlap (diagonal = unique problems per subset):")
    ipy_display.display(overlap_df)

    # warn if there's unexpected overlap
    for i, subset_a in enumerate(primary_subsets):
        for j, subset_b in enumerate(primary_subsets):
            if i < j and overlap_matrix[i, j] > 0:
                print(f"WARNING: {subset_a} and {subset_b} share {overlap_matrix[i, j]} problem IDs")

In [ ]:
# counterfactual-specific invariant checks (only applicable when using counterfactual eval strategy)
_is_counterfactual = (
    hasattr(datamodule, "config")
    and hasattr(datamodule.config, "evaluation_strategy")
    and str(datamodule.config.evaluation_strategy) == "counterfactual"
)

if samples_df.empty:
    print("skipped; no data to analyze")
elif not _is_counterfactual:
    print("skipped; not using counterfactual evaluation strategy")
else:
    # @@@@ TODO: update impl to also support keywords (when we get there)
    dm_config = datamodule.config
    eval_names = list(dm_config.eval_subset_names)
    derived_suffixes = ["_hinted", "_misleading", "_hintless"]
    for eval_name in eval_names:
        ipy_display.display(ipy_display.Markdown(f"## Counterfactual Analysis: `{eval_name}`"))
        # identify derived subset names that exist in our data
        derived_names = [
            f"{eval_name}{sfx}" for sfx in derived_suffixes if f"{eval_name}{sfx}" in samples_df["subset"].unique()
        ]
        if not derived_names:
            print(f"  no derived subsets found for '{eval_name}'; skipping")
            continue
        # -- (a) filtering/attrition summary --
        metadata = datamodule._metadata
        parent_trace_count = len(metadata.get_subset_traces(eval_name)) if eval_name in metadata.subset_traces else 0
        attrition_rows = [{"subset": eval_name + " (parent)", "traces": parent_trace_count, "samples": "n/a"}]
        for derived_name in derived_names:
            derived_info = metadata.derived_subsets.get(derived_name)
            trace_count = len(derived_info.traces) if derived_info else 0
            sample_count = len(samples_df[samples_df["subset"] == derived_name])
            attrition_rows.append(
                {
                    "subset": derived_name,
                    "traces": trace_count,
                    "samples": sample_count,
                    "retention": f"{trace_count / parent_trace_count:.1%}" if parent_trace_count > 0 else "n/a",
                }
            )
        attrition_df = pd.DataFrame(attrition_rows).set_index("subset")
        ipy_display.display(ipy_display.Markdown("### Filtering / Attrition Summary"))
        ipy_display.display(attrition_df)
        # -- (b) equal size check --
        derived_sample_counts = {name: len(samples_df[samples_df["subset"] == name]) for name in derived_names}
        unique_counts = set(derived_sample_counts.values())
        ipy_display.display(ipy_display.Markdown("### Equal Size Check"))
        if len(unique_counts) == 1:
            print(f"  PASS: all derived subsets have {unique_counts.pop()} samples")
        else:
            print(f"  FAIL: derived subset sizes differ: {derived_sample_counts}")
        # -- (c) family alignment check --
        derived_family_ids: dict[str, set[str]] = {}
        for name in derived_names:
            derived_family_ids[name] = set(samples_df[samples_df["subset"] == name]["family_id"])
        all_family_sets = list(derived_family_ids.values())
        ipy_display.display(ipy_display.Markdown("### Family Alignment Check"))
        if all_family_sets and all(fset == all_family_sets[0] for fset in all_family_sets[1:]):
            print(f"  PASS: all derived subsets share the same {len(all_family_sets[0])} families")
        else:
            for name, fset in derived_family_ids.items():
                print(f"  {name}: {len(fset)} families")
            # show symmetric differences between first subset and others
            ref_name = derived_names[0]
            for name in derived_names[1:]:
                diff = derived_family_ids[ref_name].symmetric_difference(derived_family_ids[name])
                if diff:
                    print(f"  FAIL: {ref_name} vs {name} have {len(diff)} non-shared families")
        # -- (d) exact ID non-overlap check --
        derived_exact_ids: dict[str, set[str]] = {}
        for name in derived_names:
            derived_exact_ids[name] = set(samples_df[samples_df["subset"] == name]["identifier"])
        ipy_display.display(ipy_display.Markdown("### Exact ID Non-Overlap Check"))
        overlap_found = False
        for idx_a, name_a in enumerate(derived_names):
            for idx_b, name_b in enumerate(derived_names):
                if idx_a >= idx_b:
                    continue
                overlap = derived_exact_ids[name_a] & derived_exact_ids[name_b]
                if overlap:
                    overlap_found = True
                    print(
                        f"  NOTE: {name_a} and {name_b} share {len(overlap)} exact IDs "
                        f"(expected for prompt-DB hint overlaps with ::suffix)"
                    )
        if not overlap_found:
            print("  PASS: no exact ID overlap between derived subsets")
        # -- (e) code type distribution check vs parent's code_type_prob_map --
        parent_override = dm_config.dataparser_config_overrides.get(eval_name, {})
        selection_config = parent_override.get("selection_config", {})
        expected_prob_map = selection_config.get("code_type_prob_map")
        if expected_prob_map is None and hasattr(dm_config, "_get_default_code_type_prob_map"):
            expected_prob_map = dm_config._get_default_code_type_prob_map()
        ipy_display.display(ipy_display.Markdown("### Code Type Distribution Check"))
        if not expected_prob_map:
            print("  no code_type_prob_map configured for parent; skipping distribution check")
        else:
            print(f"  expected base-type distribution (from parent's code_type_prob_map): {expected_prob_map}")
            dist_rows = []
            for name in derived_names:
                subset_df = samples_df[samples_df["subset"] == name]
                total = len(subset_df)
                if total == 0:
                    continue
                # compute actual code type distribution
                actual_counts = subset_df["code_type"].value_counts()
                actual_dist = (actual_counts / total).to_dict()
                row = {"subset": name, "total": total}
                for code_type_key in expected_prob_map:
                    row[f"expected/{code_type_key}"] = expected_prob_map[code_type_key]
                for code_type_key, fraction in sorted(actual_dist.items()):
                    row[f"actual/{code_type_key}"] = round(fraction, 4)
                dist_rows.append(row)
            if dist_rows:
                dist_df = pd.DataFrame(dist_rows).set_index("subset").fillna(0)
                ipy_display.display(dist_df)
        # -- (f) compact visual summary --
        ipy_display.display(ipy_display.Markdown("### Summary Matrix"))
        summary_rows = []
        for name in derived_names:
            summary_rows.append(
                {
                    "subset": name,
                    "samples": derived_sample_counts.get(name, 0),
                    "families": len(derived_family_ids.get(name, set())),
                    "unique_exact_ids": len(derived_exact_ids.get(name, set())),
                }
            )
        summary_df = pd.DataFrame(summary_rows).set_index("subset")
        ipy_display.display(summary_df)

In [ ]:
def _get_inner_parsers(
    parser: sample_utils.SampleBuilder | torch.utils.data.ConcatDataset,
) -> list[sample_utils.SampleBuilder]:
    """Unwrap a ConcatDataset into its inner parsers, or return a single-element list."""
    if isinstance(parser, torch.utils.data.ConcatDataset):
        return list(parser.datasets)
    return [parser]


def _aggregate_stats(
    parser: sample_utils.SampleBuilder | torch.utils.data.ConcatDataset,
) -> dict[str, typing.Any]:
    """Get aggregated stats from a parser, handling ConcatDataset transparently."""
    inner = _get_inner_parsers(parser)
    combined: dict[str, typing.Any] = {}
    for inner_parser in inner:
        if not hasattr(inner_parser, "get_stats"):
            continue
        for key, value in inner_parser.get_stats().items():
            if key not in combined:
                combined[key] = value
            elif isinstance(value, (int, float)):
                combined[key] += value
    return combined


def _get_code_type_prob_map(
    parser: sample_utils.SampleBuilder | torch.utils.data.ConcatDataset,
) -> dict[str, float] | None:
    """Extract code_type_prob_map from the first inner parser that has one."""
    for inner_parser in _get_inner_parsers(parser):
        if hasattr(inner_parser, "selection_config"):
            return inner_parser.selection_config.code_type_prob_map
    return None


if builder_map:
    if target_datamodule == "shortcuts":
        stats_rows = []
        expected_probs_map = {}
        for subset_name, builder in builder_map.items():
            stats = _aggregate_stats(builder)
            stats_rows.append({"subset": subset_name, **stats})
            prob_map = _get_code_type_prob_map(builder)
            if prob_map is not None:
                expected_probs_map[subset_name] = prob_map
        stats_df = pd.DataFrame(stats_rows).set_index("subset").fillna(0)
        ipy_display.display(stats_df)
        print(f"{stats_df.columns=}")

        # also create a df for ratio'd statistics
        prefixes_to_skip = ["orig_trace", "kept_trace", "sample_count", "summaries_count", "filtered/"]
        prefixes_to_ratio = ["selected/", "code_overrides_count"]
        ratios_df_cols = [c for c in stats_df.columns if not any(c.startswith(s) for s in prefixes_to_skip)]
        ratios_df_target_cols = [c for c in ratios_df_cols if any(c.startswith(s) for s in prefixes_to_ratio)]
        ratios_df = pd.DataFrame(stats_df[ratios_df_cols], index=stats_df.index)
        ratios_df[ratios_df_target_cols] = ratios_df[ratios_df_target_cols].div(
            stats_df["sample_count"].squeeze(), axis=0
        )
        ratios_df = ratios_df.rename(columns=lambda c: c.replace("count", "ratio"))
        # only add expected probability rows for non-derived subsets; derived subsets (e.g.
        # valid_hinted, valid_hintless, valid_misleading) have builder-level code_type_prob_maps
        # that don't reflect the actual expected output distribution (the parent eval subset's
        # prob_map controls distribution via _sample_groups_by_distribution instead)
        derived_suffixes = ("_hinted", "_misleading", "_hintless", "_with_keyword", "_without_keyword")
        subsets_with_expected_probs = [
            s
            for s in target_subsets
            if s in expected_probs_map and not any(s.endswith(sfx) for sfx in derived_suffixes)
        ]
        expected_prob_rows = [
            {f"selected/type/{input_type}": prob for input_type, prob in expected_probs_map[subset_name].items()}
            for subset_name in subsets_with_expected_probs
        ]
        ratios_df = pd.concat(
            [
                ratios_df,
                pd.DataFrame(expected_prob_rows, index=[f"{s}_expected" for s in subsets_with_expected_probs]),
            ]
        )
    else:
        # keywords datamodule: compute generic + keyword-specific stats
        stats_rows = []
        for subset_name, builder in builder_map.items():
            stats = _aggregate_stats(builder)
            stats_rows.append({"subset": subset_name, **stats})
        stats_df = pd.DataFrame(stats_rows).set_index("subset").fillna(0)
        ipy_display.display(stats_df)
        print(f"{stats_df.columns=}")

        # compute keyword-specific stats from sample tags
        keyword_stats_rows = []
        for subset_name in builder_map:
            subset_df = samples_df[samples_df["subset"] == subset_name]
            sample_count = len(subset_df)
            has_keyword_count = subset_df["has_keyword"].sum()
            # count injected/refactored samples from tags
            injected_count = sum(1 for tags in subset_df["tags"] if any("keyword_injected:1" in t for t in tags))
            refactored_count = sum(1 for tags in subset_df["tags"] if any("keyword_refactored:1" in t for t in tags))
            keyword_stats_rows.append(
                {
                    "subset": subset_name,
                    "keyword/has_keyword": has_keyword_count,
                    "keyword/injected": injected_count,
                    "keyword/refactored": refactored_count,
                    "keyword/naturally_present": has_keyword_count - injected_count,
                }
            )
        keyword_stats_df = pd.DataFrame(keyword_stats_rows).set_index("subset")
        stats_df = pd.concat([stats_df, keyword_stats_df], axis=1)
        ipy_display.display(keyword_stats_df)

        # create ratios dataframe
        prefixes_to_skip = ["orig_trace", "kept_trace", "sample_count", "summaries_count", "filtered/"]
        prefixes_to_ratio = ["selected/", "keyword/"]
        ratios_df_cols = [c for c in stats_df.columns if not any(c.startswith(s) for s in prefixes_to_skip)]
        ratios_df_target_cols = [c for c in ratios_df_cols if any(c.startswith(s) for s in prefixes_to_ratio)]
        ratios_df = pd.DataFrame(stats_df[ratios_df_cols], index=stats_df.index)
        ratios_df[ratios_df_target_cols] = ratios_df[ratios_df_target_cols].div(
            stats_df["sample_count"].squeeze(), axis=0
        )
        ratios_df = ratios_df.rename(columns=lambda c: c.replace("count", "ratio"))
    ratios_df = ratios_df.sort_index()
    ipy_display.display(ratios_df)
else:
    print("no builds created; statistics are unavailable")

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    metric_definitions = [
        ("code_line_count", "Code length (lines)", {"bins": 30}),
        ("code_target_line_span", "Target code line span", {"bins": 30}),
        ("description_word_count", "Description length (words)", {"bins": 30}),
        ("inputs_char_length", "Input args length (chars)", {"bins": 50}),
        ("expected_output_char_length", "Expected output length (chars)", {"bins": 50}),
        ("trace_step_count", "Trace step count", {"bins": 50}),
    ]
    for metric_column, metric_title, plot_kwargs in metric_definitions:
        # compute shared bin edges across all subsets for this metric
        all_values = samples_df[metric_column].dropna()
        if all_values.empty:
            continue
        bin_count = plot_kwargs.get("bins", 30)
        shared_bins = np.linspace(all_values.min(), all_values.max(), bin_count + 1)
        # display title and column labels as markdown before the figure
        if len(target_subsets) > 1:
            subset_labels = " | ".join(target_subsets)
            ipy_display.display(ipy_display.Markdown(f"**{metric_title}** -- Columns: {subset_labels}"))
        fig, axes = plt.subplots(
            1,
            len(target_subsets),
            figsize=(6 * len(target_subsets), 4),
            sharey=True,
            sharex=True,
        )
        if len(target_subsets) == 1:
            axes = [axes]
        for axis, subset_name in zip(axes, target_subsets, strict=False):
            subset_df = samples_df[samples_df["subset"] == subset_name]
            if subset_df.empty:
                axis.text(0.5, 0.5, "No samples", ha="center", va="center", transform=axis.transAxes)
                axis.set_xlabel(metric_title)
                axis.set_ylabel("Sample count")
                continue
            axis.hist(subset_df[metric_column], bins=shared_bins, color="#2a9d8f", alpha=0.85)
            axis.set_xlabel(metric_title)
            axis.set_ylabel("Sample count")
            axis.set_yscale("log")
        plt.tight_layout()
        plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # create a shared color mapping for predict types
    predict_types = sorted(samples_df["predict_type"].unique())
    color_palette = plt.cm.tab10.colors
    predict_type_colors = {pt: color_palette[idx % len(color_palette)] for idx, pt in enumerate(predict_types)}

    # left plot: predict type distribution by subset (count)
    predict_type_counts = samples_df.groupby(["subset", "predict_type"]).size().unstack(fill_value=0).sort_index(axis=1)
    bar_colors = [predict_type_colors[pt] for pt in predict_type_counts.columns]
    predict_type_counts.plot(kind="bar", ax=axes[0], color=bar_colors)
    axes[0].set_title("Predict type distribution by subset")
    axes[0].set_ylabel("Sample count")
    axes[0].tick_params(axis="x", rotation=0)
    ymax = predict_type_counts.values.max()
    axes[0].set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in axes[0].containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        axes[0].bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    # right plot: trace step count distribution by predict type (boxplot)
    boxplot_data = [samples_df[samples_df["predict_type"] == pt]["trace_step_count"].values for pt in predict_types]
    bp = axes[1].boxplot(boxplot_data, tick_labels=predict_types, patch_artist=True)
    for patch, pt in zip(bp["boxes"], predict_types, strict=False):
        patch.set_facecolor(predict_type_colors[pt])
        patch.set_alpha(0.7)
    axes[1].set_title("Trace step count by predict type")
    axes[1].set_ylabel("Trace step count")
    axes[1].set_xlabel("Predict type")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].set_yscale("log")

    plt.tight_layout()
    plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    code_type_counts = samples_df.groupby(["subset", "code_type"]).size().unstack(fill_value=0).sort_index(axis=1)
    ax = code_type_counts.plot(kind="bar", figsize=(10, 6))
    plt.title("Code type distribution by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = code_type_counts.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    tag_counter = collections.Counter()
    for tag_list in samples_df["tags"]:
        tag_counter.update(tag_list)
    most_common_tags = tag_counter.most_common(20)
    if not most_common_tags:
        print("no tags identified in the sampled data")
    else:
        tag_labels, tag_values = zip(*most_common_tags, strict=False)
        total_samples = len(samples_df)
        proportions = [v / total_samples for v in tag_values]

        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.bar(tag_labels, tag_values, color="#264653", alpha=0.9)
        ax.set_ylabel("Occurrences")
        ax.set_title("Top 20 tags across sampled data")
        ax.set_xticks(range(len(tag_labels)))
        ax.set_xticklabels(tag_labels, rotation=45, ha="right")

        percent_labels = [f"{p * 100:.1f}%" for p in proportions]
        ax.bar_label(bars, labels=percent_labels, padding=3, fontsize=9, color="#1d3557")

        ymax = max(tag_values) if tag_values else 1
        ax.set_ylim(0, ymax * 1.15)
        ax.grid(axis="y", linestyle="--", alpha=0.4)

        plt.tight_layout()
        plt.show()

In [ ]:
if samples_df.empty:
    print("skipped; no data to visualize")
else:
    override_summary = (
        samples_df.groupby(["subset", "has_code_override"])
        .size()
        .unstack(fill_value=0)
        .rename(
            columns={
                False: "no override",
                True: "with override",
            }
        )
    )
    ax = override_summary.plot(kind="bar", figsize=(8, 5))
    plt.title("Code override counts by subset")
    plt.ylabel("Sample count")
    plt.xticks(rotation=0)

    ymax = override_summary.values.max()
    ax.set_ylim(0, ymax * 1.1 if ymax > 0 else 1)
    for container in ax.containers:
        labels = [f"{int(v)}" if v > 0 else "" for v in container.datavalues]
        ax.bar_label(container, labels=labels, label_type="edge", padding=2, fontsize=9)

    plt.show()

In [ ]:
# ------------ SAMPLE VIEWER SETTINGS ------------
viewer_target_subset: str = "valid"  # which subset to pick from
viewer_sample_index: int | None = None  # specific index (0-based), or None for random
viewer_max_string_length: int = 2000  # truncate long strings beyond this length (0 = no limit)
sample_until_code_type: str | None = None  # keep randomly sampling until hitting this code type
sample_until_pred_type: str | None = None  # keep randomly sampling until hitting this pred type
sample_until_has_keyword: bool = False  # keep randomly sampling until hitting a sample w/ keyword
# -------------------------------------------------


def display_sample(
    sample: sample_utils.SampleData,
    max_string_length: int = 2000,
) -> None:
    """Display a sample's full content with nice formatting."""
    metadata_md = f"""
## Sample Metadata

| Field | Value |
|-------|-------|
| **Identifier** | `{sample.identifier}` |
| **Predict Type** | `{sample.predict_type}` |
| **Code Type** | `{sample.code_type}` |
| **Entrypoint** | `{sample.entrypoint or "(none)"}` |
| **Line Range** | {sample.first_line} → {sample.last_line} |
| **Trace Steps** | {sample.trace_step_count} |
| **Has Code Override** | {sample.has_code_override} |
| **Has Keyword** | {kw_detector.has_keyword(sample.code)} |
"""
    ipy_display.display(ipy_display.Markdown(metadata_md))
    if sample.comma_separated_tags:
        tags = [tag.strip() for tag in sample.comma_separated_tags.split(",") if tag.strip()]
        if tags:
            tags_list = "\n".join(f"- `{tag}`" for tag in tags)
            ipy_display.display(ipy_display.Markdown(f"**Tags:**\n{tags_list}"))
        else:
            ipy_display.display(ipy_display.Markdown("**Tags:** (none)"))
    else:
        ipy_display.display(ipy_display.Markdown("**Tags:** (none)"))
    if sample.complexity_metrics:
        metrics_lines = []
        for key, val in sample.complexity_metrics.items():
            if isinstance(val, float):
                metrics_lines.append(f"- **{key}:** {val:.3f}")
            else:
                metrics_lines.append(f"- **{key}:** {val}")
        metrics_md = "**Complexity Metrics:**\n" + "\n".join(metrics_lines)
        ipy_display.display(ipy_display.Markdown(metrics_md))
    ipy_display.display(ipy_display.Markdown("## Description"))
    if sample.description:
        desc_lines = sample.description.splitlines()
        desc_blockquote = "\n".join(f"> {line}" for line in desc_lines)
        ipy_display.display(ipy_display.Markdown(desc_blockquote))
    else:
        ipy_display.display(ipy_display.Markdown("> (no description)"))
    ipy_display.display(ipy_display.Markdown("## Code"))
    code_display = pyine.utils.notebooks.format_long_string(sample.code, max_string_length)
    ipy_display.display(ipy_display.Markdown(f"```python\n{code_display}\n```"))
    ipy_display.display(ipy_display.Markdown("## Inputs"))
    if sample.predict_type == "frame_variables":
        inputs_display, is_dict = pyine.utils.notebooks.format_dict_string(sample.inputs, max_string_length)
        lang = "json" if is_dict else ""
    else:
        inputs_display = pyine.utils.notebooks.format_long_string(sample.inputs, max_string_length)
        lang = ""
    ipy_display.display(ipy_display.Markdown(f"```{lang}\n{inputs_display}\n```"))
    ipy_display.display(ipy_display.Markdown("## Expected Output"))
    if sample.predict_type == "frame_variables":
        output_display, is_dict = pyine.utils.notebooks.format_dict_string(sample.expected_output, max_string_length)
        lang = "json" if is_dict else ""
    else:
        output_display = pyine.utils.notebooks.format_long_string(sample.expected_output, max_string_length)
        lang = ""
    ipy_display.display(ipy_display.Markdown(f"```{lang}\n{output_display}\n```"))


# fetch and display the sample
if not builder_map:
    print("no builders available; run the data collection cells first")
elif viewer_target_subset not in builder_map:
    print(f"subset {viewer_target_subset!r} not found; available: {list(builder_map.keys())}")
else:
    builder = builder_map[viewer_target_subset]
    if viewer_sample_index is None:
        while True:
            picked_sample_idx = random.randint(0, len(builder) - 1)
            print(f"randomly selected sample index: {picked_sample_idx}")
            sample = builder[picked_sample_idx]
            if sample_until_code_type is not None and sample.code_type != sample_until_code_type:
                continue
            if sample_until_pred_type is not None and sample.predict_type != sample_until_pred_type:
                continue
            if sample_until_has_keyword and not kw_detector.has_keyword(sample.code):
                continue
            break
    else:
        picked_sample_idx = viewer_sample_index
        if not (0 <= picked_sample_idx < len(builder)):
            raise IndexError(f"sample index {picked_sample_idx} out of range [0, {len(builder)})")
        sample = builder[picked_sample_idx]
    ipy_display.display(ipy_display.Markdown(f"# Sample Viewer: [{viewer_target_subset}] index={picked_sample_idx}"))
    display_sample(sample, max_string_length=viewer_max_string_length)